# mva_07-ポアソン重回帰分析

---

## **要点まとめ**

**Part Ⅲ：確率の最大化**

線形回帰分析では、「数値データ」から「数値データ」を予測しました。今回は、**「数値データ」** から数値でも **「回数や個数（0以上の整数）」** を予測するための **ポアソン重回帰分析（Poisson Regression）** を学びます。これにより、線形回帰、ロジスティック回帰、ポアソン回帰という3つの主要なモデルが出揃い、**「一般化線形モデル（GLM）」** としての統一的な理解が可能になります。

### **1. カウントデータの予測：なぜ線形回帰ではダメなのか？**

「1日の来店客数」や「1時間のアクセス数」のように、**0以上の整数（非負整数）** しかとらないデータを **カウントデータ** と呼びます。

| | 目的変数| 説明変数1 | ... | 説明変数d |
|---|:---:|:---:|:---:|:---:|
| **サンプル1** | $y_{1}$ (回数) | $x_{11}$  | ... | $x_{1d}$ |
| ... | ... | ... | ... | ... |
| **サンプル i** | $y_{i}$ (回数) | $x_{i1}$ | ... | $x_{id}$ |
| ... | ... | ... | ... | ... |
| **サンプルn** | $y_{n}$ (回数) | $x_{n1}$ | ... | $x_{nd}$ |

<br>
通常の線形重回帰（最小二乗法）をカウントデータに適用すると、以下の問題が生じます。

- **予測値の矛盾**: 予測値 $\hat{y}$ がマイナスになる可能性がある（客数が -3人 とは？）。
- **分散の不均一性**: カウントデータは「平均が大きいほどばらつきも大きくなる（ポアソン分布の性質）」傾向があるが、線形回帰は「分散一定」を仮定している。

<br>

### **2. ポアソン回帰モデル**

データの発生源として、ランダムな事象の回数を表す確率分布である **ポアソン分布 (Poisson Distribution)** を仮定します。

$$P(y|\lambda) = \frac{\lambda^y e^{-\lambda}}{y!}$$

ここで、分布の形状を決める唯一のパラメータ $\lambda$（平均回数）を、説明変数 $\mathbf{x}$ を使って予測するモデルを作ります。

#### **リンク関数（Link Function）**

平均回数 $\lambda$ は必ずプラスの値である必要があります。そこで、線形結合 $\mathbf{w}^\top \mathbf{x}$ を指数関数 $\exp$ で変換して $\lambda$ に結びつけます。

$$\lambda_i = \exp(w_0 + w_1 x_{i1} + \dots + w_d x_{id}) = e^{\mathbf{w}^\top \mathbf{x}_i}$$

対数をとると $\log \lambda_i = \mathbf{w}^\top \mathbf{x}_i$ となるため、この対数変換を **リンク関数** と呼びます。
このモデルにより、どんな $\mathbf{x}$ が来ても予測値 $\lambda$ は常にプラスになります。

<br>

#### **係数の解釈（Rate Ratio）**

ポアソン回帰モデルは $\lambda = \exp(w_0 + w_1 x_1 + \dots)$ なので、変数 $x_j$ が $1$ 単位増えると、予測値 $\lambda$ は $e^{w_j}$ 倍 になります。

- $w_j > 0$ ($e^{w_j} > 1$): 回数を $e^{w_j}$ 倍に増やす効果。
- $w_j < 0$ ($e^{w_j} < 1$): 回数を $e^{w_j}$ 倍に減らす効果。

<br>

### **3. 最適化：勾配の「美しい統一（GLM）」**

パラメータ $\mathbf{w}$ を決定するために、ロジスティック回帰と同様に **最尤法**（対数尤度の最大化 $\iff$ 負の対数尤度の最小化）を用います。

目的関数 $J(\mathbf{w})$（負の対数尤度）の勾配 $\nabla J(\mathbf{w})$ を計算すると、これまでに学んだモデルと驚くべき共通点が見えてきます。

| モデル | 予測値（期待値） | 勾配 $\nabla J(\mathbf{w})$ 「(予測値 - 実測値) $\times$ 入力」|
| ----- | ----- | ----- |
| **線形回帰** | $\hat{y} = \mathbf{w}^\top \mathbf{x}$ | $\sum (\hat{y}_i - y_i) \mathbf{x}_i$ |
| **ロジスティック回帰** | $p = \sigma(\mathbf{w}^\top \mathbf{x})$ | $\sum (p_i - y_i) \mathbf{x}_i$ |
| **ポアソン回帰** | $\lambda = e^{\mathbf{w}^\top \mathbf{x}}$ | $\sum (\lambda_i - y_i) \mathbf{x}_i$ |

<br>


このように、確率分布やリンク関数が異なっていても、最適なパラメータを探すための勾配は常に **「(予測値 - 実測値) $\times$ 入力」** というシンプルな形になります。
これを **一般化線形モデル（GLM）の正準リンク関数（Canonical Link Function）** の性質と呼びます。


## **Python 演習課題**

### **mva_07-A**

✅ **目的**: ポアソン回帰の勾配を計算し、勾配降下法によってパラメータが最適化され、指数関数のカーブがカウントデータに当てはまっていく様子を確認します。

#### 🖊 **【数理理解】勾配の導出と学習プロセス**

複雑に見えるポアソン分布と指数関数ですが、これらを組み合わせて微分すると、ロジスティック回帰と同様に項が打ち消し合い、劇的にシンプルな勾配式が得られます。
1変数（重み $w$, バイアス $b$; $z_i = w x_i + b$）のケースで、その導出過程を詳しく追ってみましょう。

##### **1. 準備：モデルと尤度関数**

- **モデル（予測値）**: $\lambda_i = e^{z_i} = e^{w x_i + b}$
- **尤度（確率）**: $P(y_i|\lambda_i) = \frac{\lambda_i^{y_i} e^{-\lambda_i}}{y_i!}$

これを最大化することは、対数をとった「対数尤度」を最大化することと同じです。

##### **2. 対数尤度の計算**

データ1つ分に関する対数尤度 $l_i$ を計算します。

$$l_i = \log \left( \frac{\lambda_i^{y_i} e^{-\lambda_i}}{y_i!} \right) = y_i \log \lambda_i - \lambda_i - \log y_i!$$

ここで $\lambda_i = e^{z_i}$ なので、$\log \lambda_i = z_i$ となり、非常にシンプルになります。

$$l_i = y_i z_i - e^{z_i} - (\text{定数})$$

##### **3. 勾配の導出（連鎖律）**

最小化すべき損失関数（負の対数尤度）を $J_i = -l_i = e^{z_i} - y_i z_i$ と置きます。
これをパラメータ $w$ で偏微分します。

$$\frac{\partial J_i}{\partial w} = \frac{\partial J_i}{\partial z_i} \cdot \frac{\partial z_i}{\partial w}$$

それぞれ計算してみましょう。

- (A) $z$ による微分:
$$\frac{\partial J_i}{\partial z_i} = \frac{\partial}{\partial z_i} (e^{z_i} - y_i z_i) = e^{z_i} - y_i = \lambda_i - y_i$$

- (B) 線形結合の微分: $z_i = w x_i + b$

$$\frac{\partial z_i}{\partial w} = x_i$$

##### **4. 最終的な勾配の導出**

これら(A)(B)を掛け合わせると、ロジスティック回帰と同じ形の式が現れます。

$$\frac{\partial J_i}{\partial w} = (\lambda_i - y_i) x_i$$

これを全データ $n$ 個について足し合わせれば、全体の勾配式になります。

$$\frac{\partial J}{\partial w} = \sum_{i=1}^n (\lambda_i - y_i) x_i$$

同様に、バイアス $b$ については $\frac{\partial z_i}{\partial b} = 1$ より以下の表式が得られます。

$$\frac{\partial J}{\partial b} = \sum_{i=1}^n (\lambda_i - y_i)\cdot 1$$

この **「(予測 - 正解) $\times$ 入力」** という式は、線形回帰、ロジスティック回帰、ポアソン回帰ですべて共通しています。これがGLM（一般化線形モデル）の美しい性質です。


<br>

#### **具体例で確認してみよう！**

ある商品の「広告費 ($x$)」と「販売個数 ($y$)」のデータを考え、導出した勾配式 使い、徐々に最適解へ近づいていく様子を確認してみましょう。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # 3Dプロット用
from sklearn.linear_model import PoissonRegressor
from sklearn.model_selection import train_test_split
import plotly.graph_objects as go

# ==========================================
# mva_07-A: 勾配降下法によるポアソン回帰の実装
# ==========================================
print("\n=== mva_07-A: Gradient Descent Implementation ===")

# 1. データの定義
# 広告費 (x) と 販売個数 (y: カウントデータ)
x = np.array([1, 2, 3, 4, 5, 6])
y = np.array([2, 3, 6, 9, 15, 25])  # 指数関数的に増えている

# --- 追加: データの確認 ---
# データをデータフレームにまとめて表示
df_A = pd.DataFrame({'Ad Cost (x)': x, 'Sales Count (y)': y})

print("--- 使用するデータ ---")
try:
    display(df_A) # Jupyter Notebook / Colab 環境用
except NameError:
    print(df_A)   # 通常のPythonスクリプト環境用
# ------------------------

# 2. 必要な関数の定義
def predict_lambda(w, b, x):
    """予測関数: 線形結合 -> 指数関数 (lambda > 0)"""
    # オーバーフロー対策として、入力値を制限する場合もあるが、ここではシンプルに実装
    return np.exp(w * x + b)

def calculate_gradient_poisson(w, b, x, y):
    """
    現在のパラメータ (w, b) における勾配と損失を計算する関数
    """
    # (a) 予測 (lambda)
    lam = predict_lambda(w, b, x)

    # (b) 勾配 (Gradient)
    # 解析的に導出したシンプルな式: sum( (lambda - y) * x )
    diff = lam - y          # 予測と正解のズレ (誤差)
    dw = np.sum(diff * x)   # w の勾配 (誤差 × 入力)
    db = np.sum(diff)       # b の勾配 (x=1とみなす)

    # (c) 損失 (Negative Log-Likelihood) も計算しておく（学習曲線の描画用）
    # ポアソン分布の負の対数尤度 (定数項 log(y!) は省略可だが、損失の形状を見るため主要項を計算)
    # J = sum( lambda - y * log(lambda) )
    loss = np.sum(lam - y * np.log(lam + 1e-7))

    return dw, db, loss, lam

# --- 勾配降下法による学習ループ ---

# 初期パラメータ（わざと悪い値からスタート）
w, b = 0., 0.
learning_rate = 0.001 # 学習率（発散しやすいので小さめに設定）
epochs = 2000         # 繰り返す回数

# 記録用リスト（軌跡の描画用）
history_loss = []
history_w = []
history_b = []

print(f"\nStart Training: w={w:.2f}, b={b:.2f}")

for i in range(epochs):
    # 現在の場所での勾配と損失を計算
    dw, db, loss, lam = calculate_gradient_poisson(w, b, x, y)

    # 履歴に保存
    history_loss.append(loss)
    history_w.append(w)
    history_b.append(b)

    # パラメータの更新 (勾配の逆方向へ坂を下る)
    w = w - learning_rate * dw
    b = b - learning_rate * db

    if i % 500 == 0:
        print(f"Iter {i}: Loss={loss:.4f}, w={w:.3f}, b={b:.3f}")

print(f"Final: w={w:.3f}, b={b:.3f}, Loss={loss:.4f}")

# --- 可視化1: パラメータ空間での最適化の軌跡 (3D & 2D) ---

# 1. 損失関数の等高線・曲面を描くためのグリッドデータの作成
# 軌跡が含まれる範囲より少し広めに設定
w_min, w_max = min(history_w) - 0.2, max(history_w) + 0.2
b_min, b_max = min(history_b) - 0.5, max(history_b) + 0.5

# wとbの組み合わせを作成
ww, bb = np.meshgrid(np.linspace(w_min, w_max, 50),
                     np.linspace(b_min, b_max, 50))
Z_loss = np.zeros_like(ww)

# 各グリッド点での損失を計算
for i in range(ww.shape[0]):
    for j in range(ww.shape[1]):
        w_val, b_val = ww[i, j], bb[i, j]
        lam_val = predict_lambda(w_val, b_val, x)
        # ポアソン回帰の損失関数
        loss_val = np.sum(lam_val - y * np.log(lam_val + 1e-7))
        Z_loss[i, j] = loss_val

plt.figure(figsize=(16, 7))

# (1) 左側: 3D w-b-Loss 空間上の損失曲面と軌跡
ax1 = plt.subplot(1, 2, 1, projection='3d')

# 損失曲面の描画
ax1.plot_surface(ww, bb, Z_loss, cmap='viridis', alpha=0.5, edgecolor='none')

# 軌跡を3次元的にプロット
step = 10 # 間引いてプロット
ax1.plot(history_w[::step], history_b[::step], history_loss[::step], 'r.-', markersize=3, linewidth=1, label='Trajectory', zorder=10)
# スタートとゴール
ax1.scatter(history_w[0], history_b[0], history_loss[0], s=50, c='blue', marker='o', label='Start')
ax1.scatter(history_w[-1], history_b[-1], history_loss[-1], s=100, c='red', marker='*', label='End')

ax1.set_xlabel('Weight (w)')
ax1.set_ylabel('Bias (b)')
ax1.set_zlabel('Loss')
ax1.set_title('Poisson Loss Surface & Trajectory (3D)')
ax1.view_init(elev=30, azim=120)
ax1.legend()

# (2) 右側: 2D w-b 平面上の等高線と軌跡
ax2 = plt.subplot(1, 2, 2)

# 背景に損失の等高線を描画
contour = ax2.contourf(ww, bb, Z_loss, levels=20, cmap='viridis', alpha=0.8)
plt.colorbar(contour, ax=ax2, label='Loss (Negative Log-Likelihood)')

# パラメータ更新の軌跡をプロット
ax2.plot(history_w[::step], history_b[::step], 'r.-', markersize=3, linewidth=1, label='Trajectory')
ax2.scatter(history_w[0], history_b[0], s=50, c='blue', marker='o', label='Start', edgecolors='white')
ax2.scatter(history_w[-1], history_b[-1], s=100, c='red', marker='*', label='End', edgecolors='white')

ax2.set_xlabel('Weight (w)')
ax2.set_ylabel('Bias (b)')
ax2.set_title('Gradient Descent Trajectory (2D)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

ポアソン回帰モデルによる予測曲線がデータに適合していく様子も可視化して確認しましょう。

In [ ]:
# --- 可視化2: 学習経過による指数関数カーブの変化 ---
plt.figure(figsize=(10, 6))
x_range = np.linspace(0, 7, 100)

# プロットする反復回数（インデックス）を指定
# 収束が早いため、初期状態・1回更新後・最終状態の3点を表示
step_indices = [0, 1, epochs - 1]
num_plots = len(step_indices)

# 色の準備
cmap = plt.get_cmap('coolwarm')
colors = cmap(np.linspace(0, 1, num_plots))

for i, idx in enumerate(step_indices):
    if idx >= len(history_w):
        idx = len(history_w) - 1

    w_val = history_w[idx]
    b_val = history_b[idx]

    # 線のスタイル設定
    if i == 0:
        label = f'Start (Iter {idx})'
        linestyle = '--'
        alpha = 0.5
        width = 1.5
    elif i == num_plots - 1:
        label = f'Final (Iter {idx})'
        linestyle = '-'
        alpha = 1.0
        width = 3.0
    else:
        label = f'Iter {idx}'
        linestyle = '-'
        alpha = 0.6
        width = 1.5

    # プロット
    y_pred_curve = predict_lambda(w_val, b_val, x_range)
    plt.plot(x_range, y_pred_curve,
             color=colors[i], linestyle=linestyle, linewidth=width, alpha=alpha,
             label=label)

# データ点
plt.scatter(x, y, color='black', s=100, label='Data', zorder=10)

plt.xlabel('Ad Cost (x)')
plt.ylabel('Sales Count (y)')
plt.legend()
plt.title(f'Learning Process: Exponential Curve Fitting (Epochs={epochs})')
plt.grid(True, alpha=0.3)
plt.show()


### **mva_07-B**
✅ **目的**: ``scikit-learn`` の ``PoissonRegressor`` を使用してポアソン重回帰分析を実装し、係数の意味（比率への影響）を解釈します。

#### 💻 **【実装・応用】 PoissonRegressor による実装と係数の解釈**

実務では ``scikit-learn`` の線形モデルモジュールに含まれる ``PoissonRegressor`` を使用します。

- ``scikit-learn`` 公式ドキュメント: ``PoissonRegressor``

#### **具体例で確認してみよう！**

あるECサイトの「1日のアクセス数」を、以下の2つの量的データで予測します。

- 平均気温 ($x_1$)：気温が高い/低いとアクセスはどう変化するか？
- 割引率 ($x_2$)：セール（割引率アップ）をするとアクセスは何倍になるか？



In [ ]:
# 1. データの生成 (量的データ x 量的データ)
# データ: [気温(℃), 割引率(%)] -> アクセス数
# 気温が低いほど、割引率が高いほどアクセスが増える設定
data = {
    'Temperature':   [25, 30, 15, 20, 35, 10, 28, 22, 5, 32],
    'Discount_Rate': [10,  0, 20, 30,  5, 40, 15, 10, 50, 0], # ％表記
    'Access_Count':  [150, 90, 250, 450, 80, 550, 200, 180, 800, 85]
}
df = pd.DataFrame(data)

X = df[['Temperature', 'Discount_Rate']]
y_target = df['Access_Count']

print("--- 分析データ（すべて量的データ） ---")
display(df)

# 2. モデルの学習
model = PoissonRegressor(alpha=0, max_iter=1000)
model.fit(X, y_target)

# 3. 結果の表示
w_temp = model.coef_[0]
w_disc = model.coef_[1]
intercept = model.intercept_

print("\n--- 推定パラメータ ---")
print(f"切片 (intercept): {intercept:.4f}")
print(f"気温の係数 (w1): {w_temp:.4f}")
print(f"割引率の係数 (w2): {w_disc:.4f}")

# 4. 解釈 (Rate Ratio: e^w)
rr_temp = np.exp(w_temp)
rr_disc = np.exp(w_disc)

print("\n--- 係数の解釈 (倍率) ---")
print(f"気温が1℃上がると、アクセス数は {rr_temp:.3f} 倍になる ({ (rr_temp-1)*100:.1f}% の増減)")
print(f"割引率が1%上がると、アクセス数は {rr_disc:.3f} 倍になる (+{ (rr_disc-1)*100:.1f}% の増加)")

# 補足: 割引率が10%上がった場合の影響
rr_disc_10 = np.exp(w_disc * 10)
print(f"※割引率を一気に10%上げると、アクセス数は {rr_disc_10:.2f} 倍になる")


# 5. 予測の実演
# 例: 気温20℃で、割引率が 0% の時 vs 30% の時
T = 20
D_R1 = 0
D_R2 = 30
X_new = pd.DataFrame([[T, D_R1], [T, D_R2]], columns=['Temperature', 'Discount_Rate'])
y_pred = model.predict(X_new)

print(f"\n気温{T}℃ 割引なし({D_R1}%) の予測アクセス数: {y_pred[0]:.1f} 回")
print(f"気温{T}℃ 割引あり({D_R2}%) の予測アクセス数: {y_pred[1]:.1f} 回")

In [ ]:
# ==========================================
# 参考: 3Dプロットによる可視化 (量的変数2つ)
# ==========================================

# グリッド作成 (両方の軸で連続的なメッシュを作る)
t_range = np.linspace(df['Temperature'].min(), df['Temperature'].max(), 20)
d_range = np.linspace(df['Discount_Rate'].min(), df['Discount_Rate'].max(), 20)
tt, dd = np.meshgrid(t_range, d_range)

# 予測曲面の計算
X_mesh = pd.DataFrame({'Temperature': tt.ravel(), 'Discount_Rate': dd.ravel()})
z_mesh = model.predict(X_mesh).reshape(tt.shape)

fig = go.Figure()

# 予測曲面
fig.add_trace(go.Surface(
    x=tt, y=dd, z=z_mesh,
    opacity=0.8, colorscale='Viridis', name='Prediction',
    colorbar=dict(title='Access Count')
))

# データ点
fig.add_trace(go.Scatter3d(
    x=df['Temperature'], y=df['Discount_Rate'], z=df['Access_Count'],
    mode='markers',
    marker=dict(size=5, color='red', line=dict(width=1, color='black')),
    name='Data'
))

fig.update_layout(
    title='Poisson Regression: Temp & Discount vs Access',
    scene=dict(
        xaxis_title='Temperature (C)',
        yaxis_title='Discount Rate (%)',
        zaxis_title='Access Count'
    ),
    width=800, height=600,
    margin=dict(l=0, r=0, b=0, t=40)
)

# 実行環境によっては fig.show() が機能しない場合があります
try:
    fig.show()
except:
    print("Plotly figure generated. Use a notebook environment to view interactive 3D plots.")